# Clase 14 — Práctica: Casos de uso reales (capstone)

En la clase de teoría recorrimos un flujo de trabajo completo sobre el dataset de diabetes: cargar, EDA, limpiar, enriquecer, modelar, evaluar e informar — el mismo proceso que siguió Lila en su primer proyecto real. Esta práctica es tu capstone: vas a repetir ese mismo flujo completo, pero con un objetivo distinto.

## Caso de uso: predice el índice de masa corporal en vez de la progresión de la enfermedad

En clase predijimos `target` (progresión de la diabetes) a partir de todas las demás columnas. Ahora vas a construir la misma canalización completa, pero con un objetivo distinto: predecir `bmi` (índice de masa corporal) a partir del resto de variables clínicas (sin usar `target`, que en este ejercicio no nos interesa).

Ve completando cada `# TODO` en orden. El andamiaje ya está escrito y se ejecuta sin errores tal cual está — tu trabajo es ir rellenando cada parte marcada.

**Pistas generales:**
- Puedes reutilizar casi literalmente la estructura que viste en clase, solo cambiando qué columna es el objetivo y qué columnas son las variables de entrada.
- No hace falta que "bmi" quede excluido de las variables de entrada en el paso de limpieza — solo debe quedar excluido de `X` en el paso de modelado.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

np.random.seed(42)

# Paso 1: cargar datos (ya escrito, igual que en clase)
datos = load_diabetes(as_frame=True)
df = datos.frame
print(df.shape)
df.head()

In [ ]:
# Paso 2: EDA — TODO: haz un histograma de la nueva variable objetivo, "bmi",
# y un diagrama de dispersión de "bmi" frente a alguna otra columna que creas
# que puede estar relacionada (por ejemplo "bp", la presión arterial media).

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# TODO: histograma de df["bmi"] en axes[0]

# TODO: scatter de una columna elegida por ti (eje X) frente a df["bmi"] (eje Y) en axes[1]

plt.tight_layout()
plt.show()

In [ ]:
# Paso 3: limpieza — vamos a inyectar nulos en una columna distinta a "bmi"
# esta vez, ya que "bmi" es ahora nuestro objetivo y no queremos ensuciarlo.
n_filas = len(df)
indices_nulos = np.random.choice(df.index, size=int(n_filas * 0.05), replace=False)
df.loc[indices_nulos, "bp"] = np.nan

print("Valores nulos antes de limpiar:", df["bp"].isna().sum())

# TODO: imputa los nulos de "bp" con la mediana de esa columna (fillna)

print("Valores nulos después de limpiar:", df["bp"].isna().sum())

In [ ]:
# Paso 4: feature engineering
# TODO: crea una columna derivada nueva, distinta de "indice_riesgo" visto en clase.
#       Por ejemplo, combina dos columnas cualesquiera del dataset con una suma,
#       resta o multiplicación (el objetivo es solo practicar la mecánica).
df["nueva_columna"] = 0  # <-- sustituye 0 por una combinación real de otras columnas

df.head()

In [ ]:
# Paso 5: entrenar y evaluar el modelo
# TODO: construye X (todas las columnas EXCEPTO "bmi", que ahora es el objetivo)
#       y también deberías excluir "target" de X para que no se mezcle con el ejercicio original
X = df  # <-- reemplaza esta línea: usa df.drop(columns=[...]) con las columnas correctas
y = df["bmi"]  # el nuevo objetivo

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# TODO: crea un modelo LinearRegression, entrénalo con .fit() y genera predicciones con .predict()
modelo = None
predicciones = None

# TODO: calcula mse y r2 con mean_squared_error y r2_score
mse = None
r2 = None

print(f"MSE: {mse}")
print(f"R²: {r2}")

<details>
<summary>💡 Ver solución</summary>

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

np.random.seed(42)

# Paso 1: cargar datos
datos = load_diabetes(as_frame=True)
df = datos.frame

# Paso 2: EDA
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(df["bmi"], bins=20, color="steelblue")
axes[0].set_title("Distribución de bmi")
axes[0].set_xlabel("bmi")

axes[1].scatter(df["bp"], df["bmi"], alpha=0.5, color="seagreen")
axes[1].set_title("Presión arterial (bp) vs. bmi")
axes[1].set_xlabel("bp")
axes[1].set_ylabel("bmi")
plt.tight_layout()
plt.show()

# Paso 3: limpieza
n_filas = len(df)
indices_nulos = np.random.choice(df.index, size=int(n_filas * 0.05), replace=False)
df.loc[indices_nulos, "bp"] = np.nan
df["bp"] = df["bp"].fillna(df["bp"].median())

# Paso 4: feature engineering
df["nueva_columna"] = df["s1"] + df["s2"]  # ejemplo: suma de dos medidas de sangre

# Paso 5: modelo
X = df.drop(columns=["bmi", "target"])
y = df["bmi"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

modelo = LinearRegression()
modelo.fit(X_train, y_train)
predicciones = modelo.predict(X_test)

mse = mean_squared_error(y_test, predicciones)
r2 = r2_score(y_test, predicciones)

print(f"MSE: {mse:.4f}")
print(f"R²: {r2:.2f}")
```

Fíjate en que la estructura del código es casi idéntica a la de clase — lo único que cambia realmente es qué columna juega el papel de objetivo (`y`) y cuáles juegan el papel de variables de entrada (`X`). Ese es, precisamente, el punto de este ejercicio: el mismo esqueleto de flujo de trabajo sirve para preguntas de negocio completamente distintas.
</details>

## Informe final (tu turno)

Escribe, en dos o tres frases, un pequeño informe de cierre sobre tu ejercicio: ¿qué variable parece estar más relacionada con `bmi`? ¿Qué tan bien (o mal) explica tu modelo la variación de `bmi`, según el R² obtenido? ¿Qué harías de forma distinta en un segundo intento (más datos, otro modelo, otra característica derivada)?

Este pequeño ejercicio de redacción es, en sí mismo, la habilidad más importante de todo el curso: convertir un resultado técnico en una conclusión que otra persona pueda entender y usar.

---

Vuelve a la [Clase 14 — Teoría](../notebooks_clase/clase14_casos_de_uso_reales.ipynb) o al [manual del capítulo 16](../manual/16_casos_de_uso_reales.md).

Con este ejercicio se cierra el curso completo. Enhorabuena por haber llegado hasta aquí — el mapa ya lo tienes, el resto es seguir practicando.